# DLP Insider Threat Detection — Full Pipeline
**CERT r4.2 · Isolation Forest · LSTM Autoencoder · Ground Truth Evaluation**

Run each section in order. The notebook covers:
1. Setup — mount Drive, install dependencies
2. Data cleaning — raw CSVs → daily feature matrix
3. Isolation Forest — train, score, visualize
4. LSTM Autoencoder — train on GPU, score, visualize
5. Model comparison
6. Ground truth evaluation (CERT r4.2 insiders)
7. Streamlit dashboard via public URL

> **Prerequisite:** Upload the CERT r4.2 raw CSVs to `MyDrive/dlp-project/archive/` and the answers file to `MyDrive/dlp-project/archive/answers/answers/insiders.csv` before running.

---
## 0 · Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
from pathlib import Path

# ── Set project root (adjust if your folder name differs) ──
ROOT = Path('/content/drive/MyDrive/dlp-project')
assert ROOT.exists(), f'Project folder not found at {ROOT}. Check your Drive path.'

os.environ['DLP_ROOT'] = str(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'colab'))

%cd {ROOT}
print('Working directory:', os.getcwd())

In [ ]:
# Install dependencies
!pip install -q -r colab/requirements-colab.txt
print('Dependencies installed.')

In [ ]:
# Check GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print(f'GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('No GPU found. LSTM training will be slower — consider Runtime > Change runtime type > GPU.')

# Verify required raw files
ARCHIVE = ROOT / 'archive'
required = ['email.csv', 'logon.csv', 'device.csv', 'file.csv',
            'psychometric.csv', 'users.csv', 'decoy_file.csv']
print('\nArchive file check:')
all_ok = True
for f in required:
    exists = (ARCHIVE / f).exists()
    print(f'  {f:30s} {"OK" if exists else "MISSING"}')
    if not exists:
        all_ok = False
if all_ok:
    print('All files present. Ready to proceed.')
else:
    print('Upload missing files to archive/ before continuing.')

---
## 1 · Data Cleaning

Processes `email.csv`, `logon.csv`, `device.csv`, `file.csv`, `psychometric.csv` → `cleaned/email_user_daily_with_psychometric.csv`

Features aggregated per user per day: 32 features covering email, logon, USB, file activity, and OCEAN psychometrics.

**Expected runtime: ~8–15 min** (email.csv is ~2GB, processed in 200K-row chunks)

In [ ]:
!python scripts/clean_cert_email_data.py

In [ ]:
import pandas as pd

merged = pd.read_csv(ROOT / 'cleaned/email_user_daily_with_psychometric.csv')
print(f'Daily feature matrix: {len(merged):,} rows | {merged["user"].nunique()} users')
print(f'Date range: {merged["email_day"].min()} → {merged["email_day"].max()}')
print(f'Columns ({len(merged.columns)}):', list(merged.columns))
print('\nSplit distribution:')
print(merged['dataset_split'].value_counts().to_string())
merged.head(3)

---
## 2 · Isolation Forest (Baseline)

Trains a 300-tree Isolation Forest on the training split.  
Scores every row; saves model to `models/isolation_forest_cert.pkl` and scored CSV to `cleaned/email_user_daily_scored.csv`.

**Expected runtime: ~3–5 min**

In [ ]:
!python colab/train_isolation_forest_cert.py

In [ ]:
!python colab/inference_isolation_forest_cert.py

In [ ]:
import json, matplotlib.pyplot as plt

with open(ROOT / 'models/isolation_forest_summary.json') as f:
    if_summary = json.load(f)

print('=== Isolation Forest Summary ===')
for k, v in if_summary.items():
    if k not in ('top_anomalies', 'top_feature_columns'):
        print(f'  {k}: {v}')

print('\nTop 10 anomalies:')
for a in if_summary['top_anomalies'][:10]:
    print(f'  {a["user"]}  {a["email_day"]}  score={a["iforest_score"]:.4f}  [{a["risk_severity"]}]')

In [ ]:
# Isolation Forest visualizations
!python colab/visualize_isolation_forest_cert.py

In [ ]:
from IPython.display import Image, display
import glob

for img in sorted(glob.glob(str(ROOT / 'plots/iforest_*.png'))):
    print(Path(img).name)
    display(Image(filename=img, width=900))

In [ ]:
# Interactive score table
if_df = pd.read_csv(ROOT / 'cleaned/email_user_daily_scored.csv')
print(f'Scored rows: {len(if_df):,}')
print(f'Suspicious : {(if_df["risk_severity"]=="suspicious").sum():,}')
print(f'High risk  : {(if_df["risk_severity"]=="high").sum():,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for split, color in {'train': '#4C9BE8', 'test': '#E85454'}.items():
    data = if_df[if_df['dataset_split'] == split]['iforest_score']
    axes[0].hist(data, bins=60, alpha=0.6, color=color, label=split, density=True)
axes[0].axvline(if_summary['suspicious_threshold'], color='orange', lw=2, linestyle='--', label='suspicious')
axes[0].axvline(if_summary['high_threshold'], color='red', lw=2, linestyle='--', label='high')
axes[0].set_title('IF Score Distribution'); axes[0].set_xlabel('Score'); axes[0].legend()

top = if_df.groupby('user')['iforest_score'].mean().sort_values(ascending=False).head(20)
cols = ['#E85454' if v > 0.6 else '#F4A83A' if v > 0.4 else '#4C9BE8' for v in top.values]
axes[1].barh(top.index[::-1], top.values[::-1], color=cols[::-1])
axes[1].set_title('Top 20 Users — Mean IF Score')

plt.tight_layout(); plt.show()

---
## 3 · LSTM Autoencoder

Trains a single global LSTM autoencoder (window=7 days, hidden=32, latent=16) on all users' training sequences.  
Uses GPU if available. Saves model to `models/lstm_autoencoder_cert.pkl` and scored CSV to `cleaned/email_user_daily_lstm_scored.csv`.

**Expected runtime: ~10–20 min on GPU · ~60–90 min on CPU**

In [ ]:
!python colab/train_lstm_autoencoder_cert.py

In [ ]:
!python colab/inference_lstm_autoencoder_cert.py

In [ ]:
with open(ROOT / 'models/lstm_autoencoder_summary.json') as f:
    lstm_summary = json.load(f)

print('=== LSTM Autoencoder Summary ===')
for k, v in lstm_summary.items():
    if k != 'top_anomalies':
        print(f'  {k}: {v}')

print('\nTop 10 anomalies:')
for a in lstm_summary['top_anomalies'][:10]:
    print(f'  {a["user"]}  {a["email_day"]}  score={a["lstm_score"]:.4f}  [{a["lstm_risk_severity"]}]')

In [ ]:
!python colab/visualize_lstm_autoencoder_cert.py

In [ ]:
for img in sorted(glob.glob(str(ROOT / 'plots/lstm_*.png'))):
    print(Path(img).name)
    display(Image(filename=img, width=900))

In [ ]:
lstm_df = pd.read_csv(ROOT / 'cleaned/email_user_daily_lstm_scored.csv')
lstm_scored = lstm_df[lstm_df['lstm_risk_severity'] != 'undetermined']
print(f'Scored rows (excluding undetermined): {len(lstm_scored):,}')
print(f'Suspicious : {(lstm_scored["lstm_risk_severity"]=="suspicious").sum():,}')
print(f'High risk  : {(lstm_scored["lstm_risk_severity"]=="high").sum():,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for split, color in {'train': '#4C9BE8', 'test': '#E85454'}.items():
    data = lstm_scored[lstm_scored['dataset_split'] == split]['lstm_score'].dropna()
    axes[0].hist(data, bins=60, alpha=0.6, color=color, label=split, density=True)
axes[0].axvline(lstm_summary['global_suspicious_threshold'], color='orange', lw=2, linestyle='--', label='suspicious')
axes[0].axvline(lstm_summary['global_high_threshold'], color='red', lw=2, linestyle='--', label='high')
axes[0].set_title('LSTM Score Distribution'); axes[0].set_xlabel('Score'); axes[0].legend()

top_lstm = lstm_scored.groupby('user')['lstm_score'].mean().sort_values(ascending=False).head(20)
cols = ['#E85454' if v > 0.7 else '#F4A83A' if v > 0.5 else '#4C9BE8' for v in top_lstm.values]
axes[1].barh(top_lstm.index[::-1], top_lstm.values[::-1], color=cols[::-1])
axes[1].set_title('Top 20 Users — Mean LSTM Score')

plt.tight_layout(); plt.show()

---
## 4 · Model Comparison

In [ ]:
import numpy as np

merged_scores = if_df[['user', 'email_day', 'dataset_split', 'iforest_score']].merge(
    lstm_df[['user', 'email_day', 'lstm_score']],
    on=['user', 'email_day'], how='inner'
).dropna(subset=['iforest_score', 'lstm_score'])

print(f'Rows with both scores: {len(merged_scores):,}')

# Score correlation per split
print('\nScore correlation (IF vs LSTM):')
for split in ['train', 'test']:
    sub = merged_scores[merged_scores['dataset_split'] == split]
    corr = sub['iforest_score'].corr(sub['lstm_score'])
    print(f'  {split:8s}: r = {corr:.4f}')

# Scatter: IF vs LSTM
sample = merged_scores.sample(min(6000, len(merged_scores)), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for split, color in {'train': '#4C9BE8', 'test': '#E85454'}.items():
    mask = sample['dataset_split'] == split
    axes[0].scatter(sample.loc[mask, 'iforest_score'], sample.loc[mask, 'lstm_score'],
                    alpha=0.3, s=8, color=color, label=split)
axes[0].set_xlabel('Isolation Forest Score'); axes[0].set_ylabel('LSTM Score')
axes[0].set_title('IF vs LSTM Score Scatter'); axes[0].legend()

# User-level agreement (top 10%)
user_if   = merged_scores.groupby('user')['iforest_score'].mean()
user_lstm = merged_scores.groupby('user')['lstm_score'].mean()
user_cmp  = pd.concat([user_if.rename('if_mean'), user_lstm.rename('lstm_mean')], axis=1).dropna()
user_cmp['if_flag']   = user_cmp['if_mean']   > user_cmp['if_mean'].quantile(0.90)
user_cmp['lstm_flag'] = user_cmp['lstm_mean'] > user_cmp['lstm_mean'].quantile(0.90)

both      = int((user_cmp['if_flag'] & user_cmp['lstm_flag']).sum())
only_if   = int((user_cmp['if_flag'] & ~user_cmp['lstm_flag']).sum())
only_lstm = int((~user_cmp['if_flag'] & user_cmp['lstm_flag']).sum())

axes[1].barh(['Agreement'], [both],      color='#E85454', label=f'Both ({both})')
axes[1].barh(['Agreement'], [only_if],   left=both, color='#4C9BE8', label=f'IF only ({only_if})')
axes[1].barh(['Agreement'], [only_lstm], left=both+only_if, color='#F4A83A', label=f'LSTM only ({only_lstm})')
axes[1].set_xlabel('Users'); axes[1].set_title('Top-10% Flagging Agreement')
axes[1].legend()

plt.tight_layout(); plt.show()

print(f'\nUsers flagged by BOTH models (top 10%): {both}')
top_combined = (user_cmp[user_cmp['if_flag'] & user_cmp['lstm_flag']]
                .assign(combined=lambda d: d['if_mean'] + d['lstm_mean'])
                .sort_values('combined', ascending=False)
                .head(10))
print(top_combined.round(4).to_string())

---
## 5 · Ground Truth Evaluation — CERT r4.2

Evaluates both models against the 70 known insider users from the CERT r4.2 answers file.  
Metrics: ROC AUC, Average Precision, Precision, Recall, F1 — at both day-level and user-level.

In [ ]:
ANSWERS = ROOT / 'archive/answers/answers/insiders.csv'
if not ANSWERS.exists():
    print('answers file not found at:', ANSWERS)
    print('Upload insiders.csv from the CERT download package and rerun.')
else:
    !python scripts/evaluate_models.py --answers {ANSWERS} --split test

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

# Load ground truth
ins = pd.read_csv(ANSWERS)
ins = ins[ins['dataset'] == 4.2].copy()
ins['start'] = pd.to_datetime(ins['start']).dt.normalize()
ins['end']   = pd.to_datetime(ins['end']).dt.normalize()

rows_gt = []
for _, r in ins.iterrows():
    for d in pd.date_range(r['start'], r['end'], freq='D'):
        rows_gt.append({'user': r['user'], 'email_day': d, 'is_insider': 1})
day_labels = pd.DataFrame(rows_gt).drop_duplicates()
insider_users = ins['user'].unique()
print(f'Known insider users: {len(insider_users)}')
print(f'Insider user-days  : {len(day_labels)}')

In [ ]:
# Load saved evaluation report
with open(ROOT / 'models/evaluation_report.json') as f:
    eval_report = json.load(f)

metrics_keys = ['roc_auc', 'avg_precision', 'precision', 'recall', 'f1', 'tp', 'fp', 'fn']
sections = {
    'IF — Day (test)':   eval_report.get('if_day_test',   {}),
    'IF — User (all)':   eval_report.get('if_user_all',   {}),
    'LSTM — Day (test)': eval_report.get('lstm_day_test', {}),
    'LSTM — User (all)': eval_report.get('lstm_user_all', {}),
}

rows_report = [{'Model': k, **{m: v.get(m, '-') for m in metrics_keys}}
               for k, v in sections.items()]
df_report = pd.DataFrame(rows_report).set_index('Model')
print('=== Evaluation Results ===')
print(df_report.to_string())

In [ ]:
# ROC curves
if_df['email_day']   = pd.to_datetime(if_df['email_day']).dt.normalize()
lstm_df['email_day'] = pd.to_datetime(lstm_df['email_day']).dt.normalize()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# IF ROC
test_if = if_df[if_df['dataset_split'] == 'test'].merge(
    day_labels, on=['user', 'email_day'], how='left')
test_if['is_insider'] = test_if['is_insider'].fillna(0).astype(int)
fpr, tpr, _ = roc_curve(test_if['is_insider'], test_if['iforest_score'])
auc_if = roc_auc_score(test_if['is_insider'], test_if['iforest_score'])
axes[0].plot(fpr, tpr, lw=2, color='#4C9BE8', label=f'IF (AUC={auc_if:.3f})')
axes[0].plot([0,1],[0,1],'k--',lw=1)
axes[0].set_title('Isolation Forest ROC (day-level, test)')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR'); axes[0].legend()

# LSTM ROC
test_lstm = lstm_df[(lstm_df['dataset_split'] == 'test') &
                    (lstm_df['lstm_risk_severity'] != 'undetermined')].merge(
    day_labels, on=['user', 'email_day'], how='left')
test_lstm['is_insider'] = test_lstm['is_insider'].fillna(0).astype(int)
fpr_l, tpr_l, _ = roc_curve(test_lstm['is_insider'], test_lstm['lstm_score'].fillna(0))
auc_lstm = roc_auc_score(test_lstm['is_insider'], test_lstm['lstm_score'].fillna(0))
axes[1].plot(fpr_l, tpr_l, lw=2, color='#E85454', label=f'LSTM (AUC={auc_lstm:.3f})')
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_title('LSTM Autoencoder ROC (day-level, test)')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend()

# Score distributions: insider vs normal
for label, val, color in [('Normal', 0, '#4C9BE8'), ('Insider', 1, '#E85454')]:
    axes[2].hist(test_lstm[test_lstm['is_insider']==val]['lstm_score'].dropna(),
                 bins=50, alpha=0.6, color=color, label=label, density=True)
axes[2].set_title('LSTM Score: Insider vs Normal (test)')
axes[2].set_xlabel('LSTM Score'); axes[2].legend()

plt.tight_layout(); plt.show()

In [ ]:
# Per-insider user view: which insiders did each model catch?
if_user_max = if_df.groupby('user')['iforest_score'].max().rename('if_max')
lstm_user_max = lstm_df[lstm_df['lstm_risk_severity'] != 'undetermined'].groupby('user')['lstm_score'].max().rename('lstm_max')

insider_table = (pd.DataFrame({'user': insider_users})
                 .join(if_user_max, on='user')
                 .join(lstm_user_max, on='user')
                 .fillna(0))
insider_table['if_caught']   = insider_table['if_max']   >= if_summary['suspicious_threshold']
insider_table['lstm_caught'] = insider_table['lstm_max'] >= lstm_summary['global_suspicious_threshold']
insider_table = insider_table.sort_values('if_max', ascending=False)

print(f'IF caught  : {insider_table["if_caught"].sum()} / {len(insider_users)}')
print(f'LSTM caught: {insider_table["lstm_caught"].sum()} / {len(insider_users)}')
print(f'Both caught: {(insider_table["if_caught"] & insider_table["lstm_caught"]).sum()} / {len(insider_users)}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

models  = ['Isolation Forest', 'LSTM Autoencoder']
caught  = [int(insider_table['if_caught'].sum()), int(insider_table['lstm_caught'].sum())]
missed  = [len(insider_users) - c for c in caught]
for i, (m, c, ms) in enumerate(zip(models, caught, missed)):
    axes[0].barh([m], [c],  color='#6FCF97', label='Caught' if i==0 else '')
    axes[0].barh([m], [ms], left=c, color='#E85454', label='Missed' if i==0 else '')
    axes[0].text(c/2, i, str(c), ha='center', va='center', color='white', fontweight='bold')
    axes[0].text(c+ms/2, i, str(ms), ha='center', va='center', color='white', fontweight='bold')
axes[0].set_xlabel('Insider Users'); axes[0].set_title(f'Insiders Caught vs Missed (n={len(insider_users)})')
axes[0].legend()

# IF score vs LSTM score for all insider users
axes[1].scatter(insider_table['if_max'], insider_table['lstm_max'],
                c=insider_table['if_caught'].map({True:'#E85454', False:'#4C9BE8'}),
                s=60, zorder=3)
axes[1].axvline(if_summary['suspicious_threshold'], color='orange', lw=1.5, linestyle='--', label='IF threshold')
axes[1].axhline(lstm_summary['global_suspicious_threshold'], color='red', lw=1.5, linestyle='--', label='LSTM threshold')
axes[1].set_xlabel('IF Max Score'); axes[1].set_ylabel('LSTM Max Score')
axes[1].set_title('Insider Users: IF vs LSTM Max Score')
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

print('\nTop 20 insider users by IF score:')
print(insider_table.head(20).round(4).to_string(index=False))

---
## 6 · Streamlit Dashboard (Public URL)

Launches the 7-tab monitoring dashboard accessible via a public URL using ngrok.

> **Note:** You need a free ngrok account. Get your auth token from https://dashboard.ngrok.com and paste it below.

In [ ]:
# ── Paste your ngrok token here ──────────────────────────────────────────────
NGROK_TOKEN = ''   # e.g. '2abc123_yourTokenHere'
# ─────────────────────────────────────────────────────────────────────────────

if not NGROK_TOKEN:
    print('Set NGROK_TOKEN above to launch the dashboard.')
else:
    !pip install -q streamlit pyngrok

    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN

    # Kill any existing tunnels
    ngrok.kill()

    # Start Streamlit in background
    import subprocess, threading, time

    cmd = [
        'python', '-m', 'streamlit', 'run',
        str(ROOT / 'app/monitoring_app.py'),
        '--server.port=8501',
        '--server.headless=true',
        f'--server.fileWatcherType=none',
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)   # wait for Streamlit to start

    tunnel = ngrok.connect(8501, 'http')
    print('\n' + '='*60)
    print(f'  Dashboard URL: {tunnel.public_url}')
    print('='*60)
    print('Open the URL above in your browser.')
    print('Run ngrok.kill() in a new cell to stop.')

In [ ]:
# Run this cell to stop the dashboard
# ngrok.kill()

---
## Summary

| Step | Output |
|------|--------|
| Data cleaning | `cleaned/email_user_daily_with_psychometric.csv` |
| Isolation Forest training | `models/isolation_forest_cert.pkl` |
| IF scoring | `cleaned/email_user_daily_scored.csv` |
| LSTM training | `models/lstm_autoencoder_cert.pkl` |
| LSTM scoring | `cleaned/email_user_daily_lstm_scored.csv` |
| Ground truth evaluation | `models/evaluation_report.json` |
| Visualizations | `plots/` folder |

All outputs are saved to Google Drive and persist across Colab sessions.